# Módulo 05 · Notebook 2: Consumir la API del modelo

Ya tenemos el modelo guardado (`models/house_price_model.joblib`). Ahora lo **ponemos a disposición** de otras aplicaciones: una página web, una app móvil, un Excel… Cualquiera que sepa "hablar HTTP".

```
  App web / móvil / Excel             API (FastAPI)                   Modelo
 ┌──────────────────────┐  HTTP POST  ┌─────────────────┐  predict()  ┌──────────┐
 │ {"GrLivArea": 1800,…}│ ──────────> │ valida (Pydantic)│ ──────────> │ pipeline │
 │                      │ <────────── │ responde JSON    │ <────────── │ .joblib  │
 └──────────────────────┘   200 OK    └─────────────────┘             └──────────┘
```

**Objetivos**
1. Entender el vocabulario básico: endpoint, método, JSON, código de estado.
2. Llamar a la API desde Python.
3. Ver cómo la API **valida** los datos antes de que lleguen al modelo.

> Basado en `api-example-udem-2026-01` (profesor Oscar N. Gómez).

## 0. Antes de empezar

**Opción A: con servidor (recomendada para ver la API "de verdad").** En una terminal, desde la raíz del repositorio:

```bash
uv run python 05-Pipelines-y-API/scripts/train.py        # si aún no tienes el modelo
uv run uvicorn api.main:app --reload --app-dir 05-Pipelines-y-API
```

Abre **http://127.0.0.1:8000/docs** en el navegador: FastAPI genera esa documentación interactiva automáticamente.

**Opción B: sin servidor.** Si no levantaste el servidor, este notebook usa el `TestClient` de FastAPI, que simula las llamadas HTTP en memoria. El código es casi idéntico.

In [ ]:
import sys

import requests

from intro_analitica import config

sys.path.insert(0, str(config.RAIZ / "05-Pipelines-y-API"))  # para poder importar el paquete "api"

URL = "http://127.0.0.1:8000"

try:
    requests.get(f"{URL}/health", timeout=2)
    cliente = requests.Session()
    base = URL
    print("[OK] Usando el servidor real en", URL)
except requests.ConnectionError:
    from fastapi.testclient import TestClient

    from api.main import app

    cliente = TestClient(app)
    cliente.__enter__()  # ejecuta el arranque de la app (carga del modelo)
    base = ""
    print("No hay servidor: usando TestClient (simulación en memoria)")

## 1. Vocabulario mínimo de HTTP

| Concepto | Qué es | Ejemplo |
|---|---|---|
| **Endpoint** | Una "dirección" de la API | `/predict` |
| **Método** | Qué quieres hacer | `GET` = consultar, `POST` = enviar datos |
| **JSON** | El formato de los datos | `{"OverallQual": 7}` |
| **Código de estado** | Cómo salió | `200` OK · `422` datos inválidos · `503` servicio no disponible · `500` error del servidor |

## 2. `GET /health`: ¿está viva la API?

In [ ]:
respuesta = cliente.get(f"{base}/health")
print("Código:", respuesta.status_code)
respuesta.json()

## 3. `GET /model-info`: ¿qué modelo está respondiendo?

En producción esto es oro: si las predicciones "se ven raras", lo primero es saber **qué versión** del modelo está corriendo y **cuándo** se entrenó.

In [ ]:
info = cliente.get(f"{base}/model-info").json()
{k: info[k] for k in ["modelo", "version", "entrenado_en", "metricas_test"]}

## 4. `POST /predict`: el avalúo de una casa

In [ ]:
casa = {
    "LotArea": 9000, "OverallQual": 7, "OverallCond": 5, "YearBuilt": 2005, "YearRemodAdd": 2006,
    "GrLivArea": 1800, "TotalBsmtSF": 1000, "FullBath": 2, "BedroomAbvGr": 3, "GarageCars": 2,
    "Neighborhood": "CollgCr", "BldgType": "1Fam", "HouseStyle": "2Story", "KitchenQual": "Gd",
}
respuesta = cliente.post(f"{base}/predict", json=casa)
print("Código:", respuesta.status_code)
resultado = respuesta.json()
print(f"Precio estimado: USD {resultado['precio_estimado']:,.0f}")
print(f"   Rango aproximado: USD {resultado['rango_aproximado'][0]:,.0f} - {resultado['rango_aproximado'][1]:,.0f}")

### ¿Cómo cambia el precio si mejoramos la casa?

Una API permite hacer "¿qué pasaría si…?" de forma muy sencilla: el **análisis prescriptivo** del módulo 02 hecho realidad.

In [ ]:
escenarios = {
    "Casa base": {},
    "Cocina excelente": {"KitchenQual": "Ex"},
    "+1 garaje": {"GarageCars": 3},
    "Calidad 8": {"OverallQual": 8},
    "Barrio NridgHt": {"Neighborhood": "NridgHt"},
}
base_precio = None
for nombre, cambio in escenarios.items():
    precio = cliente.post(f"{base}/predict", json={**casa, **cambio}).json()["precio_estimado"]
    base_precio = base_precio or precio
    print(f"{nombre:<18} USD {precio:>10,.0f}   ({(precio / base_precio - 1) * 100:+.1f} %)")

## 5. La validación: la API protege al modelo

¿Qué pasa si alguien envía datos sin sentido? El modelo **predeciría algo igual** (los modelos no "saben" que un área negativa es absurda).
Por eso la API valida **antes** de llamar al modelo, con las reglas definidas en `api/schemas.py`:

In [ ]:
casos_invalidos = {
    "Área negativa": {**casa, "GrLivArea": -50},
    "Calidad 15": {**casa, "OverallQual": 15},
    "Cocina 'Excelente'": {**casa, "KitchenQual": "Excelente"},
    "Remodelada antes de construida": {**casa, "YearRemodAdd": 1990},
}
for nombre, datos in casos_invalidos.items():
    r = cliente.post(f"{base}/predict", json=datos)
    detalle = r.json()["detail"]
    mensaje = detalle if isinstance(detalle, str) else detalle[0]["msg"]
    print(f"{nombre:<32} → HTTP {r.status_code}: {mensaje}")

Un **422** significa "entendí tu solicitud, pero los datos no son válidos". Fíjate que el mensaje dice **qué** está mal: eso le ahorra horas de depuración a quien use la API.

## 6. `POST /predict/batch`: muchas casas en una llamada

In [ ]:
import pandas as pd

from intro_analitica.datos import cargar_house_prices_limpio

muestra = cargar_house_prices_limpio().sample(5, random_state=config.SEMILLA)
lote = {"casas": muestra[config.FEATURES_MODELO].to_dict(orient="records")}

r = cliente.post(f"{base}/predict/batch", json=lote).json()
pd.DataFrame(
    {
        "barrio": muestra["Neighborhood"].values,
        "real": muestra["SalePrice"].values,
        "predicho": r["predicciones"],
    }
).assign(error_pct=lambda d: ((d["predicho"] / d["real"] - 1) * 100).round(1))

---
## Resumen

- La API es la **puerta de entrada** al modelo: recibe JSON, valida y responde JSON.
- `/health` y `/model-info` son para **operar** el servicio; `/predict` es para **usarlo**.
- La validación (Pydantic) es parte del producto: protege al modelo de datos absurdos.
- El **mismo** pipeline del notebook 1 hace todo el preprocesamiento: la API no repite lógica.

## Ejercicios guiados (15 min)

1. Abre `/docs` en el navegador y haz una predicción con el botón **"Try it out"**.
2. Con `curl` desde la terminal: `curl -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" -d @05-Pipelines-y-API/ejemplos/casa.json`
3. ¿Cuánto vale "tu casa soñada"? Inventa los datos y pide el avalúo. ¿Qué cambio le sube más el precio?
4. Envía 101 casas a `/predict/batch`. ¿Qué código responde? ¿Dónde está definido ese límite?

Soluciones en `ejercicios/soluciones/solucion_02_api.md`.

In [ ]:
# Tu código aquí